# Noodle biomedical literature and graph discovery
Search public literature, then retrieve one bounded citation/semantic neighborhood. No account or API key is required. Do not send patient or private case data.

In [ ]:
import json, urllib.request
from pprint import pprint
ENDPOINT = 'https://api.helena.bio/noodle/v1/mcp'
PROTOCOL = '2026-07-28'
def call_tool(name, arguments):
    body = {'jsonrpc':'2.0','id':1,'method':'tools/call','params':{'name':name,'arguments':arguments,'_meta':{'io.modelcontextprotocol/protocolVersion':PROTOCOL,'io.modelcontextprotocol/clientCapabilities':{}}}}
    request = urllib.request.Request(ENDPOINT, data=json.dumps(body).encode(), headers={'Accept':'application/json','Content-Type':'application/json','MCP-Protocol-Version':PROTOCOL,'Mcp-Method':'tools/call','Mcp-Name':name,'User-Agent':'notebook-noodle-mcp/0.1.0'}, method='POST')
    with urllib.request.urlopen(request, timeout=60) as response:
        document = json.load(response)
    if 'error' in document or document.get('result',{}).get('isError'):
        raise RuntimeError('Noodle returned a bounded tool error')
    result = document['result']
    return result.get('structuredContent', result)

In [ ]:
search = call_tool('search_biomedical_literature', {'query':'BRCA1 homologous recombination', 'limit':5, 'sort':'relevance'})
[(item.get('pmid'), item.get('title')) for item in search.get('results', [])]

In [ ]:
seed_pmid = search['results'][0]['pmid']
neighborhood = call_tool('get_publication_neighborhood', {'pmid': seed_pmid})
print('graph_version:', neighborhood.get('graph_version'))
print('nodes:', len(neighborhood.get('nodes', [])), 'edges:', len(neighborhood.get('edges', [])))
pprint(neighborhood.get('edges', [])[:2])

Preserve edge types, provenance and graph version when reporting results. Graph proximity and semantic similarity are discovery signals, not proof of causality or clinical significance.